# WM-811K — export of CNN predictions under the transformation group

This notebook loads the fitted CNN and stores its predicted probabilities for
every held-out wafer under every element of the dihedral group $D_4$. The
conformal analysis is carried out in a separate notebook, which reads these
arrays. It mirrors the EfficientNet-B0 export notebook; only the model and its
preprocessing differ.

Splitting the work this way is what makes the study cheap. The classifier is
fitted once and held fixed, so the eight forward passes are done once here; the
conformal notebook then resamples calibration/test splits of an array already in
memory.

**The transformation group.** $D_4$ consists of the four rotations by multiples
of 90 degrees and the four reflections obtained by composing them with a
horizontal flip. The wafer maps are resized to a square, so every element maps
the pixel grid onto itself exactly: no interpolation, no padding, no loss at the
borders. The group has eight elements, so the orbit average is computed exactly
rather than approximated by sampling.

Invariance is only approximate here, and deliberately so. Edge-Ring, Center,
Donut, Random and Near-full are defined by radial structure or by the absence of
directional structure, so a rotation or reflection plausibly preserves the
class; Edge-Loc, Loc and Scratch are defined by a linear trace or a localised
region, and for those the class-conditional distribution of orientations need
not be invariant. Conformal validity does not require invariance, so the case
study tests whether the departure shows up in efficiency rather than in
coverage.

## 1. Setup

In [1]:
import sys
import os
import glob

# Cerca general_utils.py nei dataset collegati al notebook
matches = glob.glob(
    "/kaggle/input/**/general_utils.py",
    recursive=True
)

print("File trovati:", matches)

if not matches:
    raise FileNotFoundError(
        "general_utils.py non trovato sotto /kaggle/input"
    )

UTILS_DIR = os.path.dirname(matches[0])
print("Cartella delle utilities:", UTILS_DIR)

if UTILS_DIR not in sys.path:
    sys.path.insert(0, UTILS_DIR)

from general_utils import (
    image_path_generation,
    get_model_probs,
    evaluate_model
)

from effnet_utils import *

print("Utilities importate correttamente.")

File trovati: ['/kaggle/input/datasets/alaurenzi/wm811k-utils/general_utils.py']
Cartella delle utilities: /kaggle/input/datasets/alaurenzi/wm811k-utils
Utilities importate correttamente.


## 2. Libraries

In [2]:
# Library
import os
import sys

import cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import (accuracy_score, roc_auc_score, recall_score,f1_score, precision_recall_fscore_support)

#BASE = ('/content/drive/MyDrive/Conformal_Prediction_Research/'
#        'Class_Conditional_CP_with_Data_Augmentation')

# Utilities from the original application, and the modules of the revision.
#sys.path.append(f'{BASE}/WMDD_application/utils')
#sys.path.append(f'{BASE}/csda_revision/code')

sys.path.append('/kaggle/input/wm811k-utils')

from general_utils import image_path_generation, evaluate_model
from image_groups import DihedralGroup

## 3. Configuration

Taken unchanged from the notebook that fitted the model, so that the
preprocessing applied here is exactly the one the network was trained with.

In [3]:
class CFG:
    """Configuration of the fitted CNN, copied from the training notebook."""
    normalize = True
    num_classes = 8
    numclasses = 8
    input_shape = (101, 101, 1)
    batch_size = 64
    label2int = {
        'Center': 0, 'Donut': 1, 'Edge-Loc': 2, 'Edge-Ring': 3,
        'Loc': 4, 'Random': 5, 'Scratch': 6, 'Near-full': 7,
    }

cfg = CFG()

## 4. Loading, transformation and prediction

Images are read in greyscale, resized to the network input, and scaled to
[0, 1]. The group element is applied to the resized image, before the channel
axis is added, so the transformation acts on the pixel grid and not on the
tensor layout.

Loading every image once and transforming the array in memory is what keeps the
export fast: the alternative, re-reading each file for each of the eight group
elements, would multiply the disk traffic by eight for no reason.

In [4]:
GROUP = DihedralGroup()          # 8 elements; the full orbit is enumerable


def load_images(df, cfg):
    """Read and preprocess every image in `df`, returning an (n, H, W) array."""
    imgs = []
    for path in tqdm(df['image_id'].values, desc="loading", ncols=90):
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise FileNotFoundError(path)
        img = cv2.resize(img, (cfg.input_shape[0], cfg.input_shape[1]))
        img = img.astype(np.float32)
        if cfg.normalize:
            img /= 255.0
        imgs.append(img)
    return np.stack(imgs)


def orbit_probs(model, images, group=GROUP, batch_size=64):
    """Predicted probabilities under every group element: shape (|G|, n, K).

    The channel axis is added after the transformation, so `group.apply` sees
    the (n, H, W) grid it expects.
    """
    out = []
    for g in group.full_orbit():
        X = group.apply(images, g)[..., None]
        out.append(model.predict(X, batch_size=batch_size, verbose=0))
    return np.stack(out)

## 5. Data

Calibration and test are merged into a single held-out set. The conformal
analysis resamples the calibration/test split at every replication, so the
variability of calibration is measured rather than fixed by one arbitrary
split. The training and validation images are not touched: the model must not
have seen any wafer used for calibration or evaluation.

In [5]:
#dataset_path = f'{BASE}/WMDD_application/dataset/'
dataset_path = '/kaggle/input/datasets/alaurenzi/wm811k-images-dataset/images/'

df = pd.read_csv(dataset_path + 'WM_811k_subset.csv', index_col=0)
df['image_id'] = df.apply(lambda row: image_path_generation(row, base_path=dataset_path), axis=1)

heldout = (df[df['set'].isin(['cal', 'test'])][['labels', 'image_id']]
             .reset_index(drop=True))
heldout['labels'] = heldout['labels'].map(cfg.label2int)

assert heldout['labels'].notna().all(), "some label is missing from cfg.label2int"
heldout['labels'] = heldout['labels'].astype(int)

INT2LABEL = {v: k for k, v in cfg.label2int.items()}
print('held-out size:', heldout.shape)
display(heldout['labels'].map(INT2LABEL).value_counts().rename('wafers').to_frame().T)

held-out size: (3649, 2)


labels,Edge-Ring,Edge-Loc,Center,Loc,Random,Scratch,Donut,Near-full
wafers,1389,635,615,413,203,202,148,44


## 6. Model

In [6]:
#model = keras.models.load_model(f'{BASE}/WMDD_application/models/CNN_best_model.keras')
model = keras.models.load_model('/kaggle/input/datasets/alaurenzi/wm811k-cnn-model/CNN_best_model.keras')
print("model loaded, input shape", model.input_shape)

I0000 00:00:1789378648.806178      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1789378648.809544      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


model loaded, input shape (None, 101, 101, 1)


## 7. Export

Three assertions guard the output. The shape must match the group size, the
number of wafers and the number of classes; the rows must be probability
vectors; and element 0 of the group, being the identity, must reproduce the
untransformed prediction. The last one is the check that the ordering of the
group elements agrees with what the conformal notebook assumes.

In [7]:
#OUT = f'{BASE}/csda_revision/data'
OUT = "/kaggle/working/csda_revision/data"
os.makedirs(OUT, exist_ok=True)

images = load_images(heldout, cfg)
labels = heldout['labels'].to_numpy().astype(np.int64)
print("images:", images.shape)

probs = orbit_probs(model, images)                      # (|G|, n, K)

plain = model.predict(images[..., None], batch_size=64, verbose=0)
assert probs.shape == (GROUP.size, len(labels), cfg.numclasses)
assert np.allclose(probs.sum(axis=2), 1.0, atol=1e-4), "rows must sum to one"
assert np.allclose(probs[0], plain, atol=1e-5), "element 0 is not the identity"

np.save(f"{OUT}/cnn_probs.npy", probs.astype(np.float32))
np.save(f"{OUT}/cnn_labels.npy", labels)
print(f"saved {probs.shape} to {OUT}")
print(f"accuracy on the held-out set: {(probs[0].argmax(1) == labels).mean():.3f}")

loading:   0%|                                                   | 0/3649 [00:00<?, ?it/s]

images: (3649, 101, 101)


I0000 00:00:1789378677.896735      71 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


saved (8, 3649, 8) to /kaggle/working/csda_revision/data
accuracy on the held-out set: 0.828


## 8. Dispersion along the orbit

How much the predicted probabilities move as the group element varies. The
theory says orbit averaging helps to the extent that the score varies along the
orbit, so this is the quantity to look at before any calibration. If the radial
classes disperse less than the directional ones, the approximate nature of the
invariance is already visible here.

In [8]:
RADIAL = ['Center', 'Donut', 'Edge-Ring', 'Random', 'Near-full']

disp = probs.std(axis=0).mean(axis=1)          # per wafer, across group elements
rows = []
for c in range(cfg.numclasses):
    m = labels == c
    name = INT2LABEL[c]
    rows.append({'class': name,
                 'symmetry': 'radial' if name in RADIAL else 'directional',
                 'n': int(m.sum()),
                 'dispersion': float(disp[m].mean())})

d = pd.DataFrame(rows).sort_values(['symmetry', 'dispersion'])
display(d.round(4))
display(d.groupby('symmetry')['dispersion'].mean().round(4).to_frame('mean'))

,class,symmetry,n,dispersion
6,Scratch,directional,202,0.0417
2,Edge-Loc,directional,635,0.0423
4,Loc,directional,413,0.0475
3,Edge-Ring,radial,1389,0.0060
7,Near-full,radial,44,0.0063
0,Center,radial,615,0.0137
1,Donut,radial,148,0.0167
5,Random,radial,203,0.0282


,mean
symmetry,
directional,0.0438
radial,0.0142


## 9. Sanity check on the fitted model

Accuracy and macro recall on the held-out set, from the untransformed
predictions. Reported to confirm that the loaded weights reproduce the
performance recorded when the model was fitted.

In [9]:
# Models to score. This notebook exports the CNN only; the EfficientNet-B0
# export writes its own array, which can be added here to score both at once.
PROBS = {'CNN': probs}
# PROBS['EffNet-B0'] = np.load(f'{OUT}/effnet_probs.npy')

cfg.tables_dir = '/kaggle/working/csda_revision/tables'
os.makedirs(cfg.tables_dir, exist_ok=True)

# The one-vs-rest AUC is undefined for a class with no held-out wafer, and
# sklearn also requires one probability column per observed class: fail here,
# with a readable message, rather than inside the loop below.
assert set(np.unique(labels)) == set(range(cfg.numclasses)), \
    "some class has no held-out wafer; the one-vs-rest AUC is undefined for it"

overall, per_class = [], []
for name, p_all in PROBS.items():
    p, p_avg = p_all[0], p_all.mean(axis=0)
    yhat, yhat_avg = p.argmax(axis=1), p_avg.argmax(axis=1)
    overall.append({'model': name,
                    'accuracy': accuracy_score(labels, yhat),
                    'macro_recall': recall_score(labels, yhat, average='macro'),
                    'macro_f1': f1_score(labels, yhat, average='macro'),
                    'auc_ovr_macro': roc_auc_score(labels, p, multi_class='ovr', average='macro'),
                    'accuracy_orbit_avg': accuracy_score(labels, yhat_avg),
                    'macro_recall_orbit_avg': recall_score(labels, yhat_avg, average='macro'),
                    'macro_f1_orbit_avg': f1_score(labels, yhat_avg, average='macro'),
                    'auc_ovr_macro_orbit_avg': roc_auc_score(labels, p_avg, multi_class='ovr', average='macro')})

    ks = np.arange(cfg.numclasses)
    prec, rec, f1, sup = precision_recall_fscore_support(labels, yhat, labels=ks, zero_division=0)
    for k in ks:
        per_class.append({'model': name, 'class': INT2LABEL[k], 'n': int(sup[k]),
                          'precision': prec[k], 'recall': rec[k], 'f1': f1[k],
                          'auc_ovr': roc_auc_score(labels == k, p[:, k])})

overall = pd.DataFrame(overall).set_index('model')
per_class = pd.DataFrame(per_class)

overall.to_csv(f'{cfg.tables_dir}/heldout_metrics.csv')
per_class.to_csv(f'{cfg.tables_dir}/heldout_metrics_per_class.csv', index=False)

display(overall.round(4))
for metric in ['recall', 'auc_ovr']:
    print(metric)
    display(per_class.pivot_table(index=['class', 'n'], columns='model', values=metric)[list(PROBS)].round(3))

,accuracy,macro_recall,macro_f1,auc_ovr_macro,accuracy_orbit_avg,macro_recall_orbit_avg,macro_f1_orbit_avg,auc_ovr_macro_orbit_avg
model,,,,,,,,
CNN,0.8282,0.7554,0.6701,0.9803,0.8465,0.7726,0.695,0.9841


recall


,model,CNN
class,n,
Center,615,0.940
Donut,148,0.980
Edge-Loc,635,0.841
Edge-Ring,1389,0.964
Loc,413,0.530
Near-full,44,0.977
Random,203,0.167
Scratch,202,0.644


auc_ovr


,model,CNN
class,n,
Center,615,0.997
Donut,148,0.998
Edge-Loc,635,0.973
Edge-Ring,1389,0.998
Loc,413,0.946
Near-full,44,0.979
Random,203,0.988
Scratch,202,0.963
